In [10]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using {device} device")

using cuda device


first, let's load our dataset and filter it down to only a few languages

In [11]:
from datasets import load_dataset

data_unfiltered = load_dataset("papluca/language-identification")
data_filtered = data_unfiltered.filter(lambda r: r["labels"] in ["en", "es", "fr", "de"])
# only look at english, spanish, french, and german
data_filtered

DatasetDict({
    train: Dataset({
        features: ['labels', 'text'],
        num_rows: 14000
    })
    validation: Dataset({
        features: ['labels', 'text'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['labels', 'text'],
        num_rows: 2000
    })
})

In [12]:
# seems pretty straightforward
data_filtered["train"][:5]

{'labels': ['es', 'de', 'de', 'es', 'es'],
 'text': ['Un producto de una calidad y capacidad increíbles que será el placer de todo amante de la tecnología',
  'Alles in allem ein super schönes Teil, deshalb die 2 Sterne! Denn: Voice Control?! Nein, ein absoluter Witz. Die reagiert nämlich nur bedingt und wenn sie gerade meint. Sprachbefehle sind, egal wie man sie ausspricht, ein Glückstreffer. Meine Freundin sagte z.B. zu mir- naja ist eben ein Weib. Daraufhin schaltete sich der Akkuträger aus bzw fragte ob ich mir sicher bin ob ich ihn ausmachen möchte.... Zusätzlich kam das Teil bei mir mit kaputtem Glastank an. Da Amazon nicht selbst der Verkäufer ist, gibt es nur die Option der Rücksendung. Schade, denn das Gerät sieht super aus und liegt schön in der Hand. Allerdings ist eben die Sprachsteuerung eine Katastrophe. Bin echt enttäuscht...',
  'Einer Freundin Geschenk da sie Flugbegleiterin ist und es gepasst hat. Allerdings hat der Anhänger nach 4-5 Wochen angefangen an den Ecken und

Let's give pytorch numerical labels to work with

In [13]:
def remap_labels(example):
    label_map = {
        "en": 0,
        "es": 1,
        "fr": 2,
        "de": 3,
    }
    example["labels"] = label_map[example["labels"]]
    return example

data = data_filtered.map(remap_labels)
data["train"][:5]

{'labels': [1, 3, 3, 1, 1],
 'text': ['Un producto de una calidad y capacidad increíbles que será el placer de todo amante de la tecnología',
  'Alles in allem ein super schönes Teil, deshalb die 2 Sterne! Denn: Voice Control?! Nein, ein absoluter Witz. Die reagiert nämlich nur bedingt und wenn sie gerade meint. Sprachbefehle sind, egal wie man sie ausspricht, ein Glückstreffer. Meine Freundin sagte z.B. zu mir- naja ist eben ein Weib. Daraufhin schaltete sich der Akkuträger aus bzw fragte ob ich mir sicher bin ob ich ihn ausmachen möchte.... Zusätzlich kam das Teil bei mir mit kaputtem Glastank an. Da Amazon nicht selbst der Verkäufer ist, gibt es nur die Option der Rücksendung. Schade, denn das Gerät sieht super aus und liegt schön in der Hand. Allerdings ist eben die Sprachsteuerung eine Katastrophe. Bin echt enttäuscht...',
  'Einer Freundin Geschenk da sie Flugbegleiterin ist und es gepasst hat. Allerdings hat der Anhänger nach 4-5 Wochen angefangen an den Ecken und Kanten braun z

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

train_texts = data["train"]["text"]
test_texts  = data["test"]["text"]
train_labels = data["train"]["labels"]
print(type(train_labels))
test_labels = data["test"]["labels"]
print(test_labels)

vectorizer = TfidfVectorizer(max_features=5000)
train_vectors = vectorizer.fit_transform(train_texts)
test_vectors  = vectorizer.transform(test_texts)

X_train = torch.tensor(train_vectors.toarray(), dtype=torch.float32)
X_test = torch.tensor(test_vectors.toarray(), dtype=torch.float32)

print(train_labels)

y_train = torch.tensor(np.array(train_labels), dtype=torch.long)
y_test = torch.tensor(np.array(test_labels), dtype=torch.long)

print(y_train)

model = nn.Sequential(
    nn.Linear(5000, 100),
    nn.ReLU(),
    nn.Linear(100, 4)
)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.1)

for epoch in range(101):
    optimizer.zero_grad() 

    logits = model(X_train) 
    loss = loss_fn(logits, y_train) 

    loss.backward() 
    optimizer.step() 

    if epoch % 10 == 0:
        print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")


with torch.no_grad():
    logits = model(X_test)
    predicted_labels = logits.argmax(dim=1) # get index of largest value in each row, dim=1 means apply to each row
    accuracy = (predicted_labels == y_test).float().mean()

print("Accuracy:", accuracy.item())

<class 'datasets.arrow_dataset.Column'>
Column([1, 2, 1, 1, 0, ...])
Column([1, 3, 3, 1, 1, ...])
tensor([1, 3, 3,  ..., 2, 1, 1])
Epoch 1, loss = 1.3872
Epoch 11, loss = 0.0001
Epoch 21, loss = 0.0000
Epoch 31, loss = 0.0000
Epoch 41, loss = 0.0000
Epoch 51, loss = 0.0000
Epoch 61, loss = 0.0000
Epoch 71, loss = 0.0000
Epoch 81, loss = 0.0000
Epoch 91, loss = 0.0000
Epoch 101, loss = 0.0000
Accuracy: 1.0


I was working with the same language dataset I looked at a few weeks ago. I filtered the dataset so that it only included four languages (English, Spanish, German, and French), the frequency of which was pretty uniformly distributed. 

The results here were...interesting, to say the least. Not only did I reduce the training time a lot, but the loss plummeted very quickly to essentially zero. If I had to guess, either a) I made a mistake somewhere (not unlikely), or b) the model picked up on unique tokens from each language (like accented characters) and emphasized those a lot. 